In [3]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import label, regionprops

# ================= CONFIG =================

IMAGE_FOLDER = r"F:\dd\apple"   # <-- CHANGE if needed
WEIGHT_FILE = r"F:\dd\Apple_Fruit_Weight_Sheet.xlsx"
OUTPUT_FILE = r"F:\dd\final_feature_extraction.xlsx"

YOLO_MODEL = "yolov8x-seg.pt"
TARGET_SIZE = 640
APPLE_CLASS_ID = 47

# ==========================================

print("Loading YOLO model...")
model = YOLO(YOLO_MODEL)

# ---------- LOAD WEIGHTS ----------

print("Loading weight file...")

weights_df = pd.read_excel(WEIGHT_FILE)

weights_df["Fruit"] = (
    weights_df["Fruit"]
    .astype(str)
    .str.strip()
    .str.lower()
)

weight_map = dict(zip(weights_df["Fruit"], weights_df["Weight"]))

print(f"Loaded {len(weight_map)} weights")

# ---------- COLLECT IMAGES (RECURSIVE) ----------

images = []

for fruit_folder in os.listdir(IMAGE_FOLDER):

    folder_path = os.path.join(IMAGE_FOLDER, fruit_folder)

    if not os.path.isdir(folder_path):
        continue

    fruit_name = fruit_folder.strip().lower()

    for img_name in os.listdir(folder_path):

        if img_name.lower().endswith((".jpg", ".jpeg", ".png")):

            full_path = os.path.join(folder_path, img_name)

            images.append((fruit_name, full_path))

print(f"Found {len(images)} total images")

# ---------- RESIZE ----------

def resize_keep_ratio(img):
    h, w = img.shape[:2]
    scale = TARGET_SIZE / max(h, w)
    return cv2.resize(img, (int(w * scale), int(h * scale)))

# ---------- SEGMENT ----------

def segment(img):

    resized = resize_keep_ratio(img)

    r = model(resized, verbose=False)[0]

    if r.masks is None:
        return None

    masks = r.masks.data.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy()

    apples = [
        masks[i]
        for i in range(len(masks))
        if int(classes[i]) == APPLE_CLASS_ID
    ]

    if not apples:
        return None

    areas = [np.sum(m > 0.5) for m in apples]
    best = apples[np.argmax(areas)]

    mask = cv2.resize(best, (img.shape[1], img.shape[0]))

    return (mask > 0.5).astype(np.uint8)

# ---------- TEXTURE FEATURES ----------

def texture_features(gray, mask):

    pixels = gray[mask == 1]

    if len(pixels) < 100:
        return [0] * 5

    pixels = (pixels / 16).astype(np.uint8)

    side = int(np.sqrt(len(pixels)))
    pixels = pixels[:side * side].reshape(side, side)

    glcm = graycomatrix(
        pixels,
        distances=[1],
        angles=[0],
        levels=16,
        symmetric=True,
        normed=True
    )

    return [
        graycoprops(glcm, p)[0, 0]
        for p in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation"]
    ]

# ---------- GEOMETRY FEATURES ----------

def geometry_features(mask):

    lab = label(mask)
    props = regionprops(lab)

    if not props:
        return None

    r = max(props, key=lambda x: x.area)

    h, w = mask.shape
    image_area = h * w

    bbox_area = (r.bbox[2] - r.bbox[0]) * (r.bbox[3] - r.bbox[1])

    return [
        r.area / image_area,
        r.eccentricity,
        r.extent,
        r.solidity,
        r.perimeter / (h + w),
        r.major_axis_length / w,
        r.minor_axis_length / h,
        r.equivalent_diameter / max(h, w),
        bbox_area / image_area
    ]


# ---------- SLICE FEATURES ----------

def slice_features(mask):

    total = np.sum(mask)

    if total == 0:
        return [0] * 20

    h, w = mask.shape
    feats = []

    for i in range(10):
        sl = mask[i * h // 10:(i + 1) * h // 10, :]
        feats.append(np.sum(sl) / total)

    for i in range(10):
        sl = mask[:, i * w // 10:(i + 1) * w // 10]
        feats.append(np.sum(sl) / total)

    return feats

# ---------- DISTRIBUTION FEATURES ----------

def distribution_features(mask):

    coords = np.column_stack(np.where(mask == 1))

    if len(coords) < 10:
        return [0] * 13

    cx, cy = coords.mean(axis=0)

    d = np.sqrt((coords[:, 0] - cx) ** 2 + (coords[:, 1] - cy) ** 2)

    return [
        np.mean(d),
        np.std(d),
        np.min(d),
        np.max(d),
        np.percentile(d, 25),
        np.percentile(d, 50),
        np.percentile(d, 75),
        np.var(d),
        np.mean(d ** 2),
        np.mean(np.abs(d - cx)),
        np.mean(np.abs(d - cy)),
        len(coords),
        np.sum(d)
    ]

# ---------- MAIN EXTRACTION ----------

print("\nStarting feature extraction...\n")

fruit_data = {}

for fruit, path in tqdm(images):

    if fruit not in weight_map:
        continue

    img = cv2.imread(path)

    if img is None:
        continue

    mask = segment(img)

    if mask is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    tex = texture_features(gray, mask)
    geo = geometry_features(mask)

    if geo is None:
        continue

    sli = slice_features(mask)
    dis = distribution_features(mask)

    feats = tex + geo + sli + dis

    fruit_data.setdefault(fruit, []).append(feats)

# ---------- AGGREGATE ----------

rows = []

for fruit, feat_list in fruit_data.items():

    arr = np.array(feat_list)
    mean_feats = np.mean(arr, axis=0)

    row = [fruit, weight_map[fruit]] + list(mean_feats)

    rows.append(row)

columns = (
    ["Fruit", "Weight",
     "Contrast", "Dissimilarity", "Homogeneity", "Energy", "Correlation",
     "Num_Pixels", "Eccentricity", "Extent", "Solidity", "Perimeter",
     "Major_Axis", "Minor_Axis", "Equiv_Diameter", "BBox_Area"] +
    [f"H_Slice_{i+1}" for i in range(10)] +
    [f"V_Slice_{i+1}" for i in range(10)] +
    [f"DF_{i+1}" for i in range(13)]
)

df = pd.DataFrame(rows, columns=columns)

df.to_excel(OUTPUT_FILE, index=False)

print("\n✅ Feature extraction complete!")
print(f"Saved: {OUTPUT_FILE}")
print(f"Total fruits processed: {len(df)}")


Loading YOLO model...
Loading weight file...
Loaded 1845 weights
Found 11058 total images

Starting feature extraction...



100%|██████████████████████████████████████████████████████████████████████████| 11058/11058 [5:29:10<00:00,  1.79s/it]



✅ Feature extraction complete!
Saved: F:\dd\final_feature_extraction.xlsx
Total fruits processed: 1843
